# Theory: Gaussian Beam Optics

**Author:** Tyler Celotta | UAF

## Abstract
The purpose of this notebook is to work through the theory and experimental design for optimizing second-harmonic generation (SHG) at 1064 nm using a periodically poled lithium niobate (PPLN) crystal (50 × 2 × 0.5 mm³). We define Gaussian beam parameters—including beam waist, Rayleigh range, divergence, and confocal parameter—to determine the optimal focusing geometry. 

The Boyd-Kleinman condition for maximum CW SHG efficiency ($L/b = 2.84$) sets a target focused waist of approximately 54.6 µm, but experimental constraints redefine a target of 77–80 µm ($L/b \approx 1.3-1.4$). 

Starting from a collimated 0.56 mm beam at 1064 nm with a measured divergence of 1.21 mRad, the collimator waist is estimated to be $\omega_{01} \approx 280$ µm, located 6.2 mm behind the collimator face. Because a waist radius of roughly 0.279 mm would result in a beam larger than the crystal itself can accommodate, we use an ABCD ray-transfer matrix to propagate the beam, constructing equations relating a chosen lens focal length $f$ to the required lens position $d_{1}$ and image distance $d_{2}$.


## 1. Experiment Specifics
* **Wavelength:** 1064 nm
* **Collimator output beam size:** 0.56 mm (1/e² diameter)
* **PPLN crystal:** 50 × 2 × 0.5 mm³
* **Previous best measured output:** 57 mW green, PPLN at 59.5°C
* **Previous calculated waist:** ~45 µm
* **Desired next target:** ~77–80 µm

In [1]:
import numpy as np
import math

# Global Parameters
wavelength = 1064e-9  # m
collimator_diam = 0.56e-3  # m
crystal_length = 50e-3  # m
measured_divergence = 1.21e-3  # rad

## 2. Definitions and Equations

**Beam Waist ($\omega_0$):** The portion of the beam where the radius is minimized. The tighter the focus, the smaller the waist, but this introduces larger divergence:
$$\omega^2(z)=\omega_0^2\left[1+\left(\frac{\lambda z}{\pi \omega_0^2}\right)^2\right]$$

**Rayleigh Range ($z_0$):** The distance at which the cross-sectional area of the beam is no more than twice that of the beam waist:
$$z_0=\frac{\pi \omega_0^2}{\lambda}$$

**Beam Divergence ($\theta$):** The far-field expansion half-angle:
$$\theta=\frac{\lambda}{\pi \omega_0}$$

**Confocal Parameter ($b$):** The total length of the focused region around the beam waist ($b = 2z_0$).

## 3. Initial Beam Parameters & Collimator Estimation

Repeating the analysis without assuming the beam waist is exactly at the collimator face (using the measured 1.21 mRad divergence).

1. Recover the true waist from the measured divergence.
2. Calculate the distance $z$ from the true waist to the collimator face where the beam radius is 0.28 mm.
3. Recalculate the true Rayleigh range.

In [2]:
# 1. True initial waist from measured divergence
omega_01 = wavelength / (np.pi * measured_divergence)
print(f"True Collimator Waist (omega_01): {omega_01 * 1e6:.1f} µm")

# 2. Distance from waist to collimator face
omega_aperture = collimator_diam / 2
# Solving the omega(z) equation for z
z_collimator = np.sqrt((omega_aperture**2 / omega_01**2) - 1) * (np.pi * omega_01**2) / wavelength
print(f"Distance from waist to face: {z_collimator * 1e3:.2f} mm")

# 3. True Rayleigh range of the initial beam
z_01 = (np.pi * omega_01**2) / wavelength
print(f"Initial Rayleigh Range (z_01): {z_01 * 100:.2f} cm")

True Collimator Waist (omega_01): 279.9 µm
Distance from waist to face: 6.11 mm
Initial Rayleigh Range (z_01): 23.13 cm


## 4. Boyd-Kleinman Focusing Targets

To recover a desired focused waist ($\omega_{02}$), we solve for the ratio $\xi \equiv L/b$. The theoretical optimum for maximum CW SHG efficiency is $L/b = 2.84$.

$$b = 2z_{02} = \frac{2\pi \omega_{02}^2}{\lambda}$$
$$\xi = \frac{L \lambda}{2\pi \omega_{02}^2}$$

In [3]:
targets_um = [45, 54.6, 77, 80] # µm

print("Boyd-Kleinman Ratios (L/b) for target waists:")
print("-" * 50)
for target in targets_um:
    omega_02 = target * 1e-6
    b = (2 * np.pi * omega_02**2) / wavelength
    xi = crystal_length / b
    print(f"Target Waist: {target:>4.1f} µm | L/b = {xi:.3f}")

Boyd-Kleinman Ratios (L/b) for target waists:
--------------------------------------------------
Target Waist: 45.0 µm | L/b = 4.181
Target Waist: 54.6 µm | L/b = 2.840
Target Waist: 77.0 µm | L/b = 1.428
Target Waist: 80.0 µm | L/b = 1.323


## 5. Lens Selection and Placement

Because the beam waist ($\omega_{01} \approx 280$ µm) is the *minimum* radius of the laser, it will scrape the inside of the 2 mm PPLN crystal if uncorrected. We must use a lens. 

Using ABCD matrix derivations for free-space and a thin lens, the equations for lens placement ($d_1$) and image distance ($d_2$) are:

**Magnification ($\alpha$):**
$$\alpha=\frac{\omega_{02}}{\omega_{01}}$$

**Distance to Lens ($d_1$):**
$$|d_1|=\sqrt{\left(\frac{f}{\alpha}\right)^2-z_{01}^2}+f$$

**Distance to Focus ($d_2$):**
$$d_2 = f + \alpha^2(d_1 - f)$$

In [ ]:
# Set your target waist and chosen lab focal length here
target_omega_02 = 80e-6  # 80 µm
focal_length = float(input("Enter the lens focal length (in mm): ")) * 1e-3 

# Calculate Magnification
alpha = target_omega_02 / omega_01

# Calculate d1 (Distance from initial waist to lens)
inside_sqrt = (focal_length / alpha)**2 - z_01**2

if inside_sqrt >= 0:
    d1 = np.sqrt(inside_sqrt) + focal_length
    # Calculate d2 (Distance from lens to new focus inside crystal)
    d2 = focal_length + (alpha**2) * (d1 - focal_length)
    
    # Calculate physical distance from the collimator face to the lens
    # Note: z_collimator was calculated in Cell 6 as a positive distance
    d1_face = d1 - z_collimator 
    
    print(f"Magnification (alpha): {alpha:.4f}")
    print(f"Lens focal length: {focal_length * 1000:.1f} mm")
    print("-" * 60)
    print(f"Optical Distance (Waist to Lens, d1): {d1 * 1000:.2f} mm")
    print(f"--> Physical Distance (Collimator FACE to Lens): {d1_face * 1000:.2f} mm")
    print("-" * 60)
    print(f"Distance from Lens to Focus (d2): {d2 * 1000:.2f} mm")
else:
    print(f"No valid real solution for a {focal_length * 1000:.1f} mm lens.")
    print("Try a lens with a longer focal length.")

Magnification (alpha): 0.2858
Lens focal length: 88.3 mm
--------------------------------------------------
Optical Distance (Waist to Lens, d1): 293.08 mm
--> Physical Distance (Collimator FACE to Lens): 286.96 mm
--------------------------------------------------
Distance from Lens to Focus (d2): 105.03 mm


## 6. Lens Selection Lab Trails

#### Trail 1:
Chosen Focal Length: $88.3$ mm

Calculated $\alpha$: $0.2858$

Calculated d1: $293.08$ mm

Physical Distance *collimator face to Lens*: $286.96$ mm

Calculated d2: $105.03$ mm

**Best Power:**


#### Trail 2:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 3:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 4:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 5:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**